# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kafkaesque-08/flyrank-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

To create a baseline action score for content refresh (Lane 3), we combine search visibility with user engagement and page age:$$\text{Action Score} = \frac{\text{impressions\_90d}}{\text{engagement\_rate} + 0.01} \times \ln(1 + \text{content\_age\_days})$$High scores prioritize pages with massive search exposure but low engagement, flagging prime candidates for editorial refresh or structural optimization.Reason CodesHIGH_IMP_LOW_ENG: High 90-day impressions but below-average engagement rate ($\le 0.3$).STALE_EVERGREEN: Content older than 365 days retaining high search volume but requiring content updates.LOW_PRIORITY: Default state for pages with low impression volume or adequate user engagement.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os
import numpy as np
import pandas as pd

# 1. Load dataset
df = pd.read_csv("flyrank-assignments/data/raw/content_refresh_anonymized.csv")

# 2. Calculate heuristic Action Score
df["baseline_action_score"] = (
    df["impressions_90d"] / (df["engagement_rate"] + 0.01)
) * np.log1p(df["content_age_days"])


# 3. Assign Reason Codes
def assign_reason_code(row):
    if row["impressions_90d"] > 1000 and row["engagement_rate"] <= 0.3:
        return "HIGH_IMP_LOW_ENG"
    elif row["content_age_days"] > 365 and row["impressions_90d"] > 500:
        return "STALE_EVERGREEN"
    else:
        return "LOW_PRIORITY"


df["reason_code"] = df.apply(assign_reason_code, axis=1)

# 4. Rank pages descending by score
df = df.sort_values(by="baseline_action_score", ascending=False).reset_index(
    drop=True
)
df["rank"] = df.index + 1

# 5. Export output CSV
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print(f"Successfully created baseline action queue at: {output_path}")
print(f"Total pages scored: {len(df):,}")

Successfully created baseline action queue at: work/outputs/baseline_action_score.csv
Total pages scored: 30,000


In [2]:
# 1. Clone your repository into this runtime
!git clone https://github.com/kafkaesque-08/flyrank-assignments.git

# 2. Re-verify where the file sits
!ls -la flyrank-assignments/data/raw/

Cloning into 'flyrank-assignments'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 138 (delta 49), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.85 MiB | 9.77 MiB/s, done.
Resolving deltas: 100% (49/49), done.
total 6580
drwxr-xr-x 2 root root    4096 Jul 21 09:17 .
drwxr-xr-x 3 root root    4096 Jul 21 09:17 ..
-rw-r--r-- 1 root root 6727670 Jul 21 09:17 content_refresh_anonymized.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# Extract top 20 items for audit
top_20 = df.head(20)[
    [
        "rank",
        "content_id",
        "content_type",
        "impressions_90d",
        "engagement_rate",
        "content_age_days",
        "baseline_action_score",
        "reason_code",
    ]
]


# Add review audit columns
def generate_audit_notes(row):
    return pd.Series(
        {
            "action": (
                "Refresh Content"
                if row["reason_code"] == "HIGH_IMP_LOW_ENG"
                else "Review Metadata"
            ),
            "confidence_note": "High search volume proves demand; low engagement flags weak content match.",
            "what_would_make_it_wrong": "The page might be an quick reference entry (e.g. glossary) where short dwell time is expected.",
        }
    )


audit_df = top_20.merge(
    top_20.apply(generate_audit_notes, axis=1), left_index=True, right_index=True
)

audit_df

,rank,content_id,content_type,impressions_90d,engagement_rate,content_age_days,baseline_action_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,keyword article,208678,0.0,362,1.230032e+08,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
1,2,content_91652435f57a,keyword article,159590,0.0,257,8.861968e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
2,3,content_080e819b7f2e,keyword article,117111,0.0,480,7.232620e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
3,4,content_11fcfd65d94c,keyword article,149083,0.0,124,7.198195e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
4,5,content_593a9691b458,keyword article,115223,0.0,466,7.081985e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
5,6,content_302dff6caa63,keyword article,113892,0.0,417,6.873931e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
6,7,content_453722754fea,keyword article,140079,0.0,97,6.422577e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
7,8,content_8bf0457e471e,keyword article,97887,0.0,480,6.045371e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
8,9,content_370de6e8e035,keyword article,114389,0.0,141,5.668921e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....
9,10,content_5d3dfb80a423,keyword article,97235,0.0,313,5.590422e+07,HIGH_IMP_LOW_ENG,Refresh Content,High search volume proves demand; low engageme...,The page might be an quick reference entry (e....


## 4. Weak picks + leakage check

Weak Picks Identified

    Glossary / Quick-Answer Pages: Pages with high impressions but low engagement rates may actually be fulfilling user intent quickly (e.g., defining a single term). Scoring them as "high priority for refresh" would waste editorial resources.

    Extreme Outliers: Unusually high impression spikes on seasonal articles can artificially inflate the score even if the content quality is fine.

Target Leakage Check

    No Future Signals: The rule strictly uses historical metrics (impressions_90d, engagement_rate, content_age_days) calculated up to the cutoff date.

    No Downstream Labels: Neither human evaluation outcomes nor future post-refresh performance flags were used in constructing the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.